# Graphic primitives

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

Graphic primitives are static decorations drawn on a plot — text labels, ellipses, arrows, and images. Use them to mark specific features in your data with prose, shapes, or screenshots.

For *reactive* annotations (markers/spans/HLines that update when the data changes), use [annotation layers](16-SciQLopAnnotationLayers.ipynb) instead.

**What you'll learn**
- Place a `Text`, `Ellipse`, `CurvedLine`, or `Pixmap` on a plot.
- Update a primitive's position, text, or styling after creation.
- Switch between data coordinates and pixel coordinates.

**Prerequisites** — [Plot panels](5-SciQLopBasicPlotPanel.ipynb).

**Next** — [Plot overlays](12-SciQLopOverlay.ipynb), [Annotation layers](16-SciQLopAnnotationLayers.ipynb).

Available primitives:
- `Text` — text labels at a position
- `Ellipse` — ellipses/circles with customizable border and fill
- `CurvedLine` — curved arrows between two points
- `Pixmap` — images placed on the plot


In [ ]:
from datetime import datetime, timezone

from SciQLop.user_api.plot import (create_plot_panel,
                                   Text, Ellipse, CurvedLine, Pixmap,
                                   LineTermination)
from SciQLop.user_api.plot.enums import CoordinateSystem
from SciQLop.user_api import TimeRange


def epoch(year, month, day, hour, minute, second=0):
    """Return a tz-aware UTC epoch float — what time-series plots want for X."""
    return datetime(year, month, day, hour, minute, second, tzinfo=timezone.utc).timestamp()


## 1. Setup: create a panel with data

First, let's plot some data to annotate.

In [ ]:
p = create_plot_panel()
p.time_range = TimeRange("2015-11-18T02:14:30", "2015-11-18T03:34:00")

plot, _ = p.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")


## 2. Adding text labels

`Text` places a label at a given `(x, y)` position in data coordinates. The x-coordinate is a Unix timestamp for time-series plots.

By default the text colour follows the plot's palette (light text on dark themes, dark text on light themes). Pass `color=`, `font_size=`, or `font_family=` to override.


In [ ]:
# Park the label well above the data (curves live in roughly [-60, +80] nT)
# so it doesn't sit on top of any trace.
label = Text(plot,
    text="Magnetopause crossing",
    x=epoch(2015, 11, 18, 2, 15), y=110,
    font_size=11,
)


In [ ]:
# Mutate after the fact — text, position, color, font_size all live on the wrapper.
label.text = "MP crossing (updated)"
label.position = (epoch(2015, 11, 18, 2, 14, 45), 105)
label.color = "#e74c3c"
label.font_size = 13


## 3. Drawing ellipses

`Ellipse` draws an ellipse defined by a bounding box `(x, y, width, height)`. Border and fill are independent: pass `line_color=`, `line_width=`, `line_style=`, `fill_color=` to the constructor, or set them as properties later.

The default border colour follows the plot palette; the fill is transparent.


In [ ]:
from PySide6.QtCore import Qt

# A thin, tight bracket-ellipse around the |B|≈0 dip — small enough to read
# the trace beneath it.
t_start = epoch(2015, 11, 18, 2, 14, 30)
t_stop = epoch(2015, 11, 18, 2, 16, 0)
ellipse = Ellipse(plot,
    x=t_start, y=-5, width=(t_stop - t_start), height=10,
    line_color="red", line_width=2.0, line_style=Qt.DashLine,
)


In [ ]:
# Tiny filled marker for a second feature — placed at a calm spot above
# the curves so it remains visible (the curves never reach +90 nT here).
t_spot = epoch(2015, 11, 18, 2, 30)
circle = Ellipse(plot,
    x=t_spot - 30, y=85, width=60, height=8,
    line_color="#4080FF", fill_color="#4080FF80",
)


## 4. Drawing curved arrows

`CurvedLine` draws a curve between two endpoints. By default it has no terminator at the start and an arrow at the stop — pass `start_termination=` / `stop_termination=` to change either. `LineTermination` is re-exported from `SciQLop.user_api.plot` and offers `NoneTermination`, `Arrow`, `LineArrow`, `SPikeArrow`, `Bar`, `HalfBar`, `SkewedBar`, `Circle`, `Diamond`, `Square`.

Like Ellipse, the default `color` follows the plot palette; pass `color=`, `line_width=`, `line_style=` to override.


In [ ]:
# Arrow from the label down to the highlighted region — stays above the
# data until the very end, where it touches the ellipse.
arrow = CurvedLine(plot,
    start=(epoch(2015, 11, 18, 2, 14, 45), 95),
    stop=(epoch(2015, 11, 18, 2, 15, 15), 5),
    color="#e74c3c", line_width=2.0,
    stop_termination=LineTermination.Arrow,
)


In [ ]:
# Pull the start handle further upward / forward in time to bend the curve
# into a gentle arc instead of a straight line.
arrow.start_direction = (epoch(2015, 11, 18, 2, 14, 55), 110)
arrow.stop_direction = (epoch(2015, 11, 18, 2, 15, 5), 25)


## 5. Visibility control

`Ellipse` and `Pixmap` expose a `.visible` property to toggle display without re-creating the primitive. `Text` and `CurvedLine` currently do not — to "hide" them, you would re-create them when needed.


In [ ]:
# Hide the ellipse
ellipse.visible = False

# Show it again
ellipse.visible = True

## 6. Coordinate systems

By default, primitives use `CoordinateSystem.Data` — positions are in data coordinates (X is a Unix timestamp for time-series, etc.) and pan/zoom with the data.

`CoordinateSystem.Pixel` switches to widget-pixel coordinates: the primitive stays fixed on screen when you pan/zoom.

> **Gotcha** — pixel coordinates are measured from the **widget** origin (top-left of the whole plot widget), and items are clipped to the **axis rect** (the plot area inside the axis labels and margins). A position like `(10, 10)` lands in the margin and gets clipped — you'll see nothing. Pick a position that's clearly inside the plot area, typically `x > ~60`, `y > ~30` after default axis margins.


In [ ]:
# A label pinned at fixed pixel coordinates — stays put when you pan/zoom.
# Position is ~80px right, ~30px down from the widget top-left, which is
# inside the plot's axis rect (i.e. not clipped by the margins).
fixed_label = Text(plot,
    text="pinned at (80, 30) px",
    x=80, y=30,
    coordinate_system=CoordinateSystem.Pixel,
)


## 7. Pixmaps — placing an image on a plot

`Pixmap` displays an image inside a bounding box (`x, y, width, height`) on the plot. The `image` argument accepts a file path (`str`), a `bytes` blob (any format `QPixmap.loadFromData` understands), or an existing `QPixmap`. Useful for embedding a satellite footprint, an instrument schematic, or an event icon.

The pixel-coordinate gotcha from §6 applies here too: position the bounding box clearly inside the axis rect (`x > ~60`, `y > ~30` after default margins), otherwise the pixmap gets clipped.


In [ ]:
# Place the SciQLop logo at a known data-space position (top-right of the
# plot, above the curves). Data coordinates are used here because
# CoordinateSystem.Pixel has a known PixmapItem ctor-order bug in
# SciQLopPlots — see backlog_pixmap_item_pixel_ctor_order.
from pathlib import Path
import SciQLop

logo = Path(SciQLop.__file__).parent / "resources" / "icons" / "SciQLop.png"

# 5-minute-wide × 20 nT logo, anchored above the curves near the end of the panel.
logo_t = epoch(2015, 11, 18, 3, 25)
logo_pixmap = Pixmap(plot,
    x=logo_t, y=110, width=300, height=15,
    image=str(logo),
    coordinate_system=CoordinateSystem.Data,
)


## 8. Lifecycle

Primitives are owned by the plot; dropping the Python reference does **not** automatically remove the graphic. To hide an `Ellipse` or `Pixmap`, set `.visible = False`. `Text` and `CurvedLine` do not yet expose `.visible` — re-create them when needed. To clear everything at once, delete the plot or panel.

## API reference

| Primitive | Required args | Optional ctor kwargs | Properties |
|-----------|---------------|----------------------|------------|
| `Text(plot, text, x, y)` | `text`, `x`, `y` | `color`, `font_size`, `font_family`, `coordinate_system` | `.text`, `.position`, `.color`, `.font`, `.font_size`, `.font_family` |
| `Ellipse(plot, x, y, w, h)` | `x`, `y`, `width`, `height` | `line_color`, `line_width`, `line_style`, `fill_color`, `coordinate_system`, `tool_tip` | `.visible`, `.line_color`, `.line_width`, `.line_style`, `.fill_color`, `.position` |
| `CurvedLine(plot, start, stop)` | `start`, `stop` | `color`, `line_width`, `line_style`, `start_termination`, `stop_termination`, `start_direction`, `stop_direction`, `coordinate_system` | `.start`, `.stop`, `.start_direction`, `.stop_direction`, `.color`, `.line_width`, `.line_style`, `.start_termination`, `.stop_termination` |
| `Pixmap(plot, x, y, w, h, image)` | `x`, `y`, `width`, `height`, `image` (str path / bytes / `QPixmap`) | `coordinate_system` | `.visible`, `.position` |

**Sensible defaults.**
- `Text`, `Ellipse`, `CurvedLine` use the plot's palette text colour by default — visible on both light and dark themes. Pass an explicit `color=` / `line_color=` to override.
- `CurvedLine`'s Bezier control handles (`start_direction`, `stop_direction`) default to 1/3 and 2/3 along the straight start → stop segment, so a freshly-constructed curve is essentially a straight line. Pull either handle off-axis to bend it.

**Termination shapes.** `LineTermination` (importable from `SciQLop.user_api.plot`) values: `NoneTermination`, `Arrow`, `LineArrow`, `SPikeArrow`, `Bar`, `HalfBar`, `SkewedBar`, `Circle`, `Diamond`, `Square`.

**Coordinate system.** All primitives accept `coordinate_system=` — either `CoordinateSystem.Data` (default) for data-space positions or `CoordinateSystem.Pixel` for widget-space positions that stay fixed on screen when you pan/zoom.
